# Demo 1 — Consume open MCP servers

Point one client at two independent, public MCP servers — the **LangChain Docs** MCP and the **AWS Knowledge** MCP — and hand their tools to an agent. The agent answers using tools it never wrote, from two unrelated providers, unified by one protocol.

*Needs network to both hosted (no-auth) servers and an `OPENAI_API_KEY` in `.env`.*

In [1]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_mcp_adapters.client import MultiServerMCPClient

In [2]:
load_dotenv()

True

## One client, two open servers
`get_tools()` connects to both and returns their tools as LangChain tools — discovered at runtime, no bespoke client code per provider.

In [3]:
client = MultiServerMCPClient(
    {
        "langchain-docs": {
            "transport": "streamable_http",
            "url": "https://docs.langchain.com/mcp",
        },
        "aws-knowledge": {
            "transport": "streamable_http",
            "url": "https://knowledge-mcp.global.api.aws",
        },
    }
)
tools = await client.get_tools()
print(f"{len(tools)} tools discovered from 2 servers:")
for t in tools:
    print(" -", t.name)

8 tools discovered from 2 servers:
 - search_docs_by_lang_chain
 - query_docs_filesystem_docs_by_lang_chain
 - submit_feedback
 - aws___read_documentation
 - aws___search_documentation
 - aws___list_regions
 - aws___get_regional_availability
 - aws___retrieve_skill


## Build an agent on the discovered tools
We wrote none of these tools. The agent just gets a tool list.

In [4]:
provider = os.getenv("LLM_PROVIDER", "openai")
model_kwargs = {"model": "gpt-4o-mini", "temperature": 0}
if provider == "vocareum":
    model_kwargs |= {"base_url": "https://openai.vocareum.com/v1", "api_key": os.getenv("VOCAREUM_API_KEY")}
agent = create_agent(ChatOpenAI(**model_kwargs), tools)

## Ask a LangChain question → routes to the LangChain-docs tool

In [7]:
res = await agent.ainvoke(
    {
        "messages": [
            {"role": "system", "content": "You are a helpful assistant that answers questions about LangChain concisely."},
            {"role": "system", "content": "Filter the results to only include Python examples (include Python in the query text)"},
            {"role": "system", "content": "Make sure to include the correct references to the LangChain documentation."},
            {"role": "user", "content": "In LangChain, what does create_agent() do? Search the docs and provide a Python example."},
        ]
    }
)
print(res["messages"][-1].content)

The `create_agent()` function in LangChain is used to create an agent that can perform tasks based on the tools and models provided. It allows you to define a system prompt and specify the tools the agent can use to fulfill user requests.

### Python Example

Here’s a Python example of how to use `create_agent()`:

```python
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

# Define low-level API tools (stubbed)
@tool
def create_calendar_event(title: str, start_time: str, end_time: str, attendees: list[str], location: str = "") -> str:
    """Create a calendar event."""
    return f"Event created: {title} from {start_time} to {end_time} with {len(attendees)} attendees"

@tool
def send_email(to: list[str], subject: str, body: str, cc: list[str] = []) -> str:
    """Send an email via email API."""
    return f"Email sent to {', '.join(to)} - Subject: {subject}"

# Initialize the chat model
model = init_chat_model("gpt-5.5")

# Create the agent
c

## Ask an AWS question → routes to the AWS-knowledge tool

In [8]:
res = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "List a few AWS regions using the AWS knowledge tools."}]}
)
print(res["messages"][-1].content)

Here are a few AWS regions:

1. **Africa (Cape Town)** - af-south-1
2. **Asia Pacific (Tokyo)** - ap-northeast-1
3. **Europe (Frankfurt)** - eu-central-1
4. **US East (N. Virginia)** - us-east-1
5. **South America (Sao Paulo)** - sa-east-1

If you need more information or additional regions, feel free to ask!


Two providers, zero tool code, one protocol.